# 02 — Model detective (Solution)

**Time:** 70 minutes
**Question:** when does the dataset contain enough information to distinguish
a sphere from an ellipsoid?

The file name deliberately reveals nothing about the generating model. Split
the work: one partner advocates for a sphere, the other for an ellipsoid.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import sans_fitter
from sans_fitter import SANSFitter

DATA = ROOT / 'data'
print(f'Using sans_fitter from {Path(sans_fitter.__file__).resolve()}')
%matplotlib inline


In [ ]:
filename = DATA / 'mystery_shape.csv'
values = np.genfromtxt(filename, delimiter=',', names=True)
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(values['Q'], values['I'], yerr=values['dI'], fmt='o', ms=3)
ax.set(xscale='log', yscale='log', xlabel=r'$q$ / Å$^{-1}$', ylabel=r'$I(q)$')
ax.grid(alpha=0.25)
plt.show()


## Exercise 1 — Competing explanations

You now know the full workflow, so this notebook gives less code. Implement a
reusable sphere fit first. Keep SLD values fixed and vary only radius, scale,
and background. Compare residual patterns, not just the numerical objective.

In [ ]:
def fit_sphere(qmin=None, qmax=None):
    fitter = SANSFitter()
    fitter.load_data(str(filename))
    if qmin is not None or qmax is not None:
        fitter.set_q_range(qmin=qmin, qmax=qmax)
    fitter.set_model('sphere')
    fitter.set_param('sld', value=4.2, vary=False)
    fitter.set_param('sld_solvent', value=1.0, vary=False)
    fitter.set_param('radius', value=45, min=10, max=180, vary=True)
    fitter.set_param('scale', value=0.04, min=0.001, max=0.2, vary=True)
    fitter.set_param('background', value=0.002, min=0, max=0.02, vary=True)
    return fitter, fitter.fit(engine='lmfit', method='least_squares')


sphere, sphere_result = fit_sphere()
sphere.plot_results(show_residuals=True, log_scale=True)


### A competing model

Create a **new** `SANSFitter` for the ellipsoid so the two fits remain
independent. After `set_model('ellipsoid')`, use `get_params()` to discover the
model's parameter names. Decide whether each should be fixed or varied.

In [ ]:
ellipsoid = SANSFitter()
ellipsoid.load_data(str(filename))
ellipsoid.set_model('ellipsoid')
ellipsoid.set_param('sld', value=4.2, vary=False)
ellipsoid.set_param('sld_solvent', value=1.0, vary=False)
ellipsoid.set_param('radius_polar', value=100, min=15, max=220, vary=True)
ellipsoid.set_param('radius_equatorial', value=40, min=10, max=120, vary=True)
ellipsoid.set_param('scale', value=0.04, min=0.001, max=0.2, vary=True)
ellipsoid.set_param('background', value=0.002, min=0, max=0.02, vary=True)
ellipsoid_result = ellipsoid.fit(engine='lmfit', method='least_squares')
ellipsoid.plot_results(show_residuals=True, log_scale=True)


In [ ]:
print('Sphere objective:', sphere_result['chisq'])
print('Ellipsoid objective:', ellipsoid_result['chisq'])
for name in ['radius_polar', 'radius_equatorial']:
    print(name, ellipsoid_result['parameters'][name]['formatted'])


## Exercise 2 — Remove information

Repeat both fits using only `0.004 ≤ q ≤ 0.025 Å⁻¹`. Predict what will happen
before running the cells. Which shape features have disappeared from view?

In [ ]:
sphere_short, sphere_short_result = fit_sphere(qmin=0.004, qmax=0.025)
sphere_short.plot_results(show_residuals=True, log_scale=True)

ellipsoid_short = SANSFitter()
ellipsoid_short.load_data(str(filename))
ellipsoid_short.set_q_range(qmin=0.004, qmax=0.025)
ellipsoid_short.set_model('ellipsoid')
ellipsoid_short.set_param('sld', value=4.2, vary=False)
ellipsoid_short.set_param('sld_solvent', value=1.0, vary=False)
ellipsoid_short.set_param('radius_polar', value=100, min=15, max=220, vary=True)
ellipsoid_short.set_param('radius_equatorial', value=40, min=10, max=120, vary=True)
ellipsoid_short.set_param('scale', value=0.04, min=0.001, max=0.2, vary=True)
ellipsoid_short.set_param('background', value=0.002, min=0, max=0.02, vary=True)
ellipsoid_short_result = ellipsoid_short.fit(engine='lmfit', method='least_squares')
ellipsoid_short.plot_results(show_residuals=True, log_scale=True)


## Verdict

Write a short referee-style verdict. Include one reason based on residuals, one
based on parameter plausibility, and one follow-up measurement that would make
the distinction stronger.

In [ ]:
conclusion = (
    'The ellipsoid is preferred on the full q-range: its residuals lose the '
    'systematic oscillatory pattern left by the sphere. After truncation the '
    'models are harder to distinguish, showing that fit quality depends on '
    'the information measured, not only the optimizer.'
)
print(conclusion)
